# 04 — Memory Optimization Experiment

Compare four training configurations to see how each one reduces VRAM:

| Config | Technique |
|--------|-----------|
| A | Baseline (BF16, no tricks) |
| B | Gradient checkpointing |
| C | 8-bit AdamW optimizer |
| D | QLoRA (4-bit + LoRA) |

We measure **peak VRAM** for each and plot the results.

In [ ]:
import sys
sys.path.append("..")

import gc
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

from src.llm_optimization.core import load_config, DataConfig
from src.llm_optimization.data import load_and_prepare_data, QADataset
from src.llm_optimization.training import (
    build_mixed_precision_trainer,
    build_gradient_checkpointing_trainer,
    build_qlora_trainer,
)
from src.llm_optimization.optimization import build_8bit_optimizer
from src.llm_optimization.core.config import OptimizerConfig
from src.llm_optimization.utils import ResourceMonitor, gpu_mem, peak_gpu_mem, reset_peak_gpu_mem

In [ ]:
def free_gpu():
    """Free CUDA cache, run GC, and reset peak tracker."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        reset_peak_gpu_mem()

In [ ]:
base = load_config('./configs/qlora.yaml')
train_df, _, val_df = load_and_prepare_data(base.data)

tokenizer = AutoTokenizer.from_pretrained(base.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, base.data.max_length)
val_ds = QADataset(val_df, tokenizer, base.data.max_length)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

## Config A — Baseline (BF16 mixed precision, no extra memory tricks)

In [ ]:
free_gpu()

mp_cfg = load_config('./configs/mixed_precision.yaml')
mp_cfg = mp_cfg.__class__(**{**mp_cfg.__dict__, "data": base.data})

trainer_a, model_a = build_mixed_precision_trainer(mp_cfg, train_ds, val_ds, tokenizer)
trainer_a.train()
vram_a = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f'\n[A] Baseline peak VRAM: {vram_a:.0f} MB')

del trainer_a, model_a
free_gpu()

## Config B — Gradient Checkpointing

Discards intermediate activations during the forward pass and recomputes them
during the backward pass. Trades ~30% compute for ~50-70% lower activation memory.

In [ ]:
free_gpu()

gc_cfg = load_config('./configs/gradient_checkpointing.yaml')
gc_cfg = gc_cfg.__class__(**{**gc_cfg.__dict__, "data": base.data})

trainer_b, model_b = build_gradient_checkpointing_trainer(gc_cfg, train_ds, val_ds, tokenizer)
trainer_b.train()
vram_b = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f'\n[B] Gradient checkpointing peak VRAM: {vram_b:.0f} MB')

del trainer_b, model_b
free_gpu()

## Config C — 8-bit AdamW optimizer

Stores optimizer moments in 8-bit instead of 32-bit → ~75% optimizer-memory savings.

In [ ]:
free_gpu()

trainer_c, model_c = build_mixed_precision_trainer(mp_cfg, train_ds, val_ds, tokenizer)

opt_c = build_8bit_optimizer(
    model_c,
    OptimizerConfig(learning_rate=float(mp_cfg.training.learning_rate)),
    use_paged=True,
)
trainer_c.optimizer = opt_c

trainer_c.train()
vram_c = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f'\n[C] 8-bit optimizer peak VRAM: {vram_c:.0f} MB')

del trainer_c, model_c, opt_c
free_gpu()

## Config D — QLoRA (4-bit base + LoRA adapters)

Combines 4-bit weight storage, double quantization, and 16-bit LoRA adapters.
The most memory-efficient configuration.

In [ ]:
free_gpu()

trainer_d, model_d = build_qlora_trainer(base, train_ds, val_ds, tokenizer)
trainer_d.train()
vram_d = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f'\n[D] QLoRA peak VRAM: {vram_d:.0f} MB')

del trainer_d, model_d
free_gpu()

## Compare results

In [ ]:
configs = ["Baseline", "Grad Checkpoint", "8-bit Optim", "QLoRA"]
vrams = [vram_a, vram_b, vram_c, vram_d]

plt.figure(figsize=(10, 5))
bars = plt.bar(configs, vrams, color=["gray", "steelblue", "seagreen", "darkorange"])
plt.ylabel("Peak VRAM (MB)")
plt.title("Peak GPU Memory During Training")
plt.grid(axis="y", alpha=0.3)

for bar, v in zip(bars, vrams):
    plt.text(
        bar.get_x() + bar.get_width() / 2, v + 20,
        f"{v:.0f} MB", ha="center", fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
df = pd.DataFrame({
    "Config":   configs,
    "Peak VRAM (MB)": [round(v, 1) for v in vrams],
})
df["Reduction vs Baseline (%)"] = (
    100 * (1 - df["Peak VRAM (MB)"] / max(df["Peak VRAM (MB)"].iloc[0], 1))
).round(1)

print(df.to_string(index=False))

# Save for later comparison
Path("../outputs").mkdir(exist_ok=True)
df.to_csv("../outputs/memory_comparison.csv", index=False)
print("\n Output saved")

## Summary

- **Gradient checkpointing** reduces *activation* memory (biggest win for long sequences).
- **8-bit optimizer** reduces *optimizer state* memory (fixed ratio, independent of batch).
- **QLoRA** reduces *all three* (weights + gradients + optimizer states) simultaneously.
- The most aggressive savings come from **combining** these techniques.

**Next:** `05_inference_benchmark.ipynb` for serving-side evaluation.